# Ukrainian Morphology FST
## Finite State Transducer for Ukrainian

**Rules implemented:**
1. **Case 1** — Noun Genitive Singular (`держава→держави`, `криниця→криниці`)
2. **Case 2** — Adjective Masculine → Feminine (`незалежний→незалежна`, `майбутній→майбутня`)
3. **Case 3** — Verb Infinitive → 1sg Present (`відчувати→відчуваю`, `хвалити→хвалю`)
4. **Rule S** — Substitution г→з in Dative (`перемога→перемозі`)
5. **Rule I** — Insertion ε→ся reflexive (`навчати→навчатися`)
6. **Rule D** — Deletion -ти→ε verb stem (`відчувати→відчува`)
7. **Complex** — Consonant alternation exceptions (`голосити→голошу`, `загубити→загублю`)


## 1. Load Lexicon

In [ ]:
lexicon = []
with open('lexicon.txt', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#'):
            lexicon.append(line.split())
print(f'Loaded {len(lexicon)} lexicon entries')
for e in lexicon[:6]:
    print(' ', e)


## 2. FST Core Classes

Each FST encodes a **suffix rewrite rule** as a sequence of arcs:

```
q0 --[σ:σ]--> q0        self-loop: copy every stem character
q0 --[s1:t1]--> q1  ┐
q1 --[s2:t2]--> q2  │   suffix arcs
...                  │
qN  :  FINAL STATE   ┘
```

- **σ** = any character (stem copy)
- **ε** on input = insertion; **ε** on output = deletion


In [ ]:
class Arc:
    """One arc in an FST: (from_state) --[input:output]--> (to_state)."""
    def __init__(self, from_state, to_state, inp, out):
        self.from_state = from_state
        self.to_state   = to_state
        self.inp        = inp
        self.out        = out

    def __repr__(self):
        return f'  ({self.from_state}) --[{self.inp}:{self.out}]--> ({self.to_state})'


class MorphFST:
    """
    Finite State Transducer for a single morphological rule.

    Encoding pattern (suffix rewrite):
      q0 --[σ:σ]--> q0          self-loop: copy every stem character
      q0 --[s1:t1]--> q1  ┐
      q1 --[s2:t2]--> q2  │  consume old suffix / emit new suffix
      ...                  │
      qN : final state     ┘
    """

    def __init__(self, name, rule_type, description):
        self.name        = name
        self.rule_type   = rule_type
        self.description = description
        self.arcs        = []
        self.states      = {'q0'}
        self.initial     = 'q0'
        self.finals      = set()
        self._suffix_in  = None
        self._suffix_out = None
        self._condition  = None

    def encode(self, suffix_in, suffix_out, condition=None):
        """Register the suffix rewrite suffix_in → suffix_out and populate arcs."""
        self._suffix_in  = suffix_in
        self._suffix_out = suffix_out
        self._condition  = condition

        # Self-loop: copy stem characters (any symbol σ)
        self.arcs.append(Arc('q0', 'q0', 'σ', 'σ'))

        n_in  = len(suffix_in)
        n_out = len(suffix_out)
        span  = max(n_in, n_out, 1)

        prev = 'q0'
        for i in range(span):
            nxt     = f'q{i+1}'
            in_sym  = suffix_in[i]  if i < n_in  else 'ε'
            out_sym = suffix_out[i] if i < n_out else 'ε'
            self.arcs.append(Arc(prev, nxt, in_sym, out_sym))
            self.states.add(nxt)
            prev = nxt

        self.finals.add(prev)
        return self

    def apply(self, word):
        """Transduce word through this FST; return result or None."""
        if self._suffix_in is None:
            return None
        if self._condition and not self._condition(word):
            return None
        if self._suffix_in == '':          # pure insertion
            return word + self._suffix_out
        if word.endswith(self._suffix_in):
            stem = word[: -len(self._suffix_in)]
            return stem + self._suffix_out
        return None

    def show(self):
        print(f"\n{'═'*62}")
        print(f'  FST : {self.name}')
        print(f'  Type: {self.rule_type}')
        print(f'  Rule: {self.description}')
        print(f'  States : {sorted(self.states)}')
        print(f'  Initial: {self.initial}   Finals: {self.finals}')
        print('  Arcs:')
        for arc in self.arcs:
            print(f'    {arc}')
        print(f"{'═'*62}")


def test(fst, word, expected):
    result = fst.apply(word)
    ok = '✓' if result == expected else '✗'
    print(f'    {ok}  {word:20s} →  {str(result):20s}  (expected: {expected})')

print('FST classes defined: Arc, MorphFST, test')


## 3. FST Instances

All 11 FST objects are built here once and reused in every section below.

In [ ]:
# ── Case 1: Noun Genitive Singular ──────────────────────────────────────
noun_gen_hard = MorphFST('noun_gen_hard', 'inflection',
                          'Noun Genitive Sg – hard stem: -а → -и').encode('а', 'и')

noun_gen_soft = MorphFST('noun_gen_soft', 'inflection',
                          'Noun Genitive Sg – soft stem: -я → -і').encode('я', 'і')

# ── Case 2: Adjective Masculine → Feminine ───────────────────────────────
adj_fem_hard = MorphFST('adj_fem_hard', 'inflection',
                         'Adjective Feminine – hard stem: -ий → -а').encode('ий', 'а')

adj_fem_soft = MorphFST('adj_fem_soft', 'inflection',
                         'Adjective Feminine – soft stem: -ій → -я').encode('ій', 'я')

# ── Case 3: Verb Infinitive → 1sg Present ───────────────────────────────
verb_1sg_aty = MorphFST('verb_1sg_aty', 'inflection',
                         'Verb 1sg Present – -ати/-яти: -ти → -ю').encode('ти', 'ю')

verb_1sg_yty = MorphFST('verb_1sg_yty', 'inflection',
                         'Verb 1sg Present – -ити: -ити → -ю').encode('ити', 'ю')

# ── Rule S: Substitution г→з in Dative ──────────────────────────────────
noun_dat_subst = MorphFST('noun_dat_subst', 'substitution',
                           'Substitution г→з in Dative: -га → -зі').encode('га', 'зі')

# ── Rule I: Insertion ε→ся (reflexive) ──────────────────────────────────
verb_reflexive = MorphFST('verb_reflexive', 'insertion',
                           'Insertion ε→ся: infinitive → reflexive (-ти → -тися)').encode('ти', 'тися')

# ── Rule D: Deletion -ти→ε (verb stem) ──────────────────────────────────
verb_stem = MorphFST('verb_stem', 'deletion',
                     'Deletion -ти→ε: strip infinitive suffix to get stem').encode('ти', '')

# ── Complex: Consonant Alternation в 1sg Present ─────────────────────────
verb_alt_s_sh = MorphFST('verb_alt_s_sh', 'inflection',
                          'Complex – consonant alternation с→ш: -сити → -шу').encode('сити', 'шу')

verb_alt_b_bl = MorphFST('verb_alt_b_bl', 'inflection',
                          'Complex – consonant alternation б→бл: -бити → -блю').encode('бити', 'блю')

print('All 11 FST instances created successfully')


## 4. Case 1 — Noun Genitive Singular (Родовий відмінок однини)

Feminine nouns of the 1st declension:

| Stem | Nominative ending | Genitive ending | Examples |
|------|------------------|-----------------|----------|
| Hard | -а               | -и              | держава→держави, людина→людини, хвилина→хвилини, справа→справи, садиба→садиби, влада→влади |
| Soft | -я               | -і              | криниця→криниці, лікарня→лікарні, пустеля→пустелі, знаряддя→знарядді, праця→праці, пекерня→пекерні |

**FST arcs (hard `-а → -и`):**
```
q0 --[σ:σ]--> q0   (copy stem)
q0 --[а:и]--> q1   FINAL
```


In [ ]:
noun_gen_hard.show()
noun_gen_soft.show()


In [ ]:
print('Hard stem (-а → -и):')
test(noun_gen_hard, 'держава',  'держави')   # state
test(noun_gen_hard, 'людина',   'людини')    # person
test(noun_gen_hard, 'хвилина',  'хвилини')   # wave / minute
test(noun_gen_hard, 'справа',   'справи')    # matter / affair
test(noun_gen_hard, 'садиба',   'садиби')    # homestead
test(noun_gen_hard, 'влада',    'влади')     # power / authority
print('Soft stem (-я → -і):')
test(noun_gen_soft, 'криниця',  'криниці')   # well
test(noun_gen_soft, 'лікарня',  'лікарні')   # hospital
test(noun_gen_soft, 'пустеля',  'пустелі')   # desert
test(noun_gen_soft, 'знаряддя', 'знарядді')  # tool / implement
test(noun_gen_soft, 'праця',    'праці')     # work / labour
test(noun_gen_soft, 'пекерня',  'пекерні')   # bakery


## 5. Case 2 — Adjective Masculine → Feminine (Прикметник чол. → жін. рід)

| Stem | Masc. ending | Fem. ending | Examples |
|------|-------------|-------------|----------|
| Hard | -ий         | -а          | незалежний→незалежна, відважний→відважна, загадковий→загадкова, оксамитовий→оксамитова, бурштиновий→бурштинова, зухвалий→зухвала |
| Soft | -ій         | -я          | майбутній→майбутня, сусідній→сусідня, справжній→справжня, безкрайній→безкрайня, всесвітній→всесвітня, довколишній→довколишня |

**FST arcs (hard `-ий → -а`):**
```
q0 --[σ:σ]--> q0
q0 --[и:а]--> q1
q1 --[й:ε]--> q2   FINAL
```


In [ ]:
adj_fem_hard.show()
adj_fem_soft.show()


In [ ]:
print('Hard stem (-ий → -а):')
test(adj_fem_hard, 'незалежний',  'незалежна')   # independent
test(adj_fem_hard, 'відважний',   'відважна')    # brave
test(adj_fem_hard, 'загадковий',  'загадкова')   # mysterious
test(adj_fem_hard, 'оксамитовий', 'оксамитова')  # velvety
test(adj_fem_hard, 'бурштиновий', 'бурштинова')  # amber
test(adj_fem_hard, 'зухвалий',    'зухвала')     # audacious
print('Soft stem (-ій → -я):')
test(adj_fem_soft, 'майбутній',    'майбутня')    # future
test(adj_fem_soft, 'сусідній',     'сусідня')     # neighbouring
test(adj_fem_soft, 'справжній',    'справжня')    # genuine
test(adj_fem_soft, 'безкрайній',   'безкрайня')   # boundless
test(adj_fem_soft, 'всесвітній',   'всесвітня')   # world-wide
test(adj_fem_soft, 'довколишній',  'довколишня')  # surrounding


## 6. Case 3 — Verb Infinitive → 1sg Present (Інфінітив → 1 ос. одн. теп. часу)

| Verb class | Rule       | Examples |
|-----------|-----------|----------|
| -ати/-яти  | -ти → -ю  | відчувати→відчуваю, перемагати→перемагаю, зупиняти→зупиняю, кохати→кохаю, мріяти→мріяю, надихати→надихаю |
| -ити       | -ити → -ю | цінити→ціню, хвалити→хвалю, молити→молю |

**FST arcs (`-ти → -ю`):**
```
q0 --[σ:σ]--> q0
q0 --[т:ю]--> q1
q1 --[и:ε]--> q2   FINAL
```


In [ ]:
verb_1sg_aty.show()
verb_1sg_yty.show()


In [ ]:
print('-ати/-яти verbs (-ти → -ю):')
test(verb_1sg_aty, 'відчувати',  'відчуваю')   # to feel
test(verb_1sg_aty, 'перемагати', 'перемагаю')  # to prevail
test(verb_1sg_aty, 'зупиняти',   'зупиняю')    # to stop
test(verb_1sg_aty, 'кохати',     'кохаю')      # to love
test(verb_1sg_aty, 'мріяти',     'мріяю')      # to dream
test(verb_1sg_aty, 'надихати',   'надихаю')    # to inspire
print('-ити verbs (-ити → -ю):')
test(verb_1sg_yty, 'цінити',     'ціню')       # to value
test(verb_1sg_yty, 'хвалити',    'хвалю')      # to praise
test(verb_1sg_yty, 'молити',     'молю')       # to pray / plead


## 7. Rule S — Substitution: г → з in Dative (Давальний відмінок)

Feminine nouns ending in **-га** mutate г→з when forming the Dative (suffix **-і** replaces **-а**).

| Nominative | Dative    |
|-----------|----------|
| перемога   | перемозі |
| наснага    | насназі  |
| звага      | звазі    |
| розвага    | розвазі  |
| вимога     | вимозі   |
| знемога    | знемозі  |

**FST arcs (`-га → -зі`):**
```
q0 --[σ:σ]--> q0
q0 --[г:з]--> q1   (substitution г → з)
q1 --[а:і]--> q2   FINAL
```


In [ ]:
noun_dat_subst.show()


In [ ]:
print('Substitution г→з (Dative):')
test(noun_dat_subst, 'перемога', 'перемозі')   # victory
test(noun_dat_subst, 'наснага',  'насназі')    # vigour / drive
test(noun_dat_subst, 'звага',    'звазі')      # courage
test(noun_dat_subst, 'розвага',  'розвазі')    # entertainment
test(noun_dat_subst, 'вимога',   'вимозі')     # requirement
test(noun_dat_subst, 'знемога',  'знемозі')    # exhaustion


## 8. Rule I — Insertion: ε → ся (Reflexive verbs / Зворотні дієслова)

Reflexive verbs are formed by **inserting** the suffix **-ся** after -ти.
In FST terms this is an **epsilon transition on input** (nothing → с, nothing → я).

| Infinitive  | Reflexive      |
|------------|----------------|
| навчати     | навчатися      |
| намагати    | намагатися     |
| піклувати   | піклуватися    |

**FST arcs (`-ти → -тися`):**
```
q0 --[σ:σ]--> q0
q0 --[т:т]--> q1
q1 --[и:и]--> q2
q2 --[ε:с]--> q3   (insert с)
q3 --[ε:я]--> q4   FINAL  (insert я)
```


In [ ]:
verb_reflexive.show()


In [ ]:
print('Insertion ε→ся (reflexive verbs):')
test(verb_reflexive, 'навчати',   'навчатися')    # to study
test(verb_reflexive, 'намагати',  'намагатися')   # to strive
test(verb_reflexive, 'піклувати', 'піклуватися')  # to take care


## 9. Rule D — Deletion: -ти → ε (Verb Stem Extraction)

Strip the infinitive suffix **-ти** to obtain the bare verb stem.
Both **т** and **и** map to **ε** (deleted from output).

| Infinitive   | Stem        |
|-------------|------------|
| відчувати    | відчува    |
| співати      | співа      |
| обіймати     | обійма     |
| надихати     | надиха     |
| допомагати   | допомага   |
| захищати     | захища     |

**FST arcs (`-ти → ε`):**
```
q0 --[σ:σ]--> q0
q0 --[т:ε]--> q1   (delete т)
q1 --[и:ε]--> q2   FINAL  (delete и)
```


In [ ]:
verb_stem.show()


In [ ]:
print('Deletion -ти→ε (verb stem):')
test(verb_stem, 'відчувати',  'відчува')    # to feel
test(verb_stem, 'співати',    'співа')      # to sing
test(verb_stem, 'обіймати',   'обійма')     # to embrace
test(verb_stem, 'надихати',   'надиха')     # to inspire
test(verb_stem, 'допомагати', 'допомага')   # to help
test(verb_stem, 'захищати',   'захища')     # to protect


## 10. Complex Case — Consonant Alternation in 1sg Present (Exceptions)

Some **-ити** verbs have consonant mutations before -ю that the regular rule cannot handle.

### Pattern с → ш  (`-сити → -шу`)
| Infinitive  | 1sg Present | Note |
|------------|-------------|------|
| голосити    | голошу      | regular rule gives *голосю ← wrong |
| виносити    | виношу      | regular rule gives *виносю ← wrong |
| заносити    | заношу      | regular rule gives *заносю ← wrong |

**FST arcs (`-сити → -шу`):**
```
q0 --[σ:σ]--> q0
q0 --[с:ш]--> q1   (substitution с → ш)
q1 --[и:у]--> q2
q2 --[т:ε]--> q3
q3 --[и:ε]--> q4   FINAL
```

### Pattern б → бл  (`-бити → -блю`)
| Infinitive  | 1sg Present | Note |
|------------|-------------|------|
| загубити    | загублю     | regular rule gives *загубю ← wrong |
| ослабити    | ослаблю     | regular rule gives *ослабю ← wrong |
| зрубити     | зрублю      | regular rule gives *зрубю ← wrong |
| доробити    | дороблю     | regular rule gives *доробю ← wrong |

**FST arcs (`-бити → -блю`):**
```
q0 --[σ:σ]--> q0
q0 --[б:б]--> q1
q1 --[и:л]--> q2   (и → л  :  drop и, insert л)
q2 --[т:ю]--> q3
q3 --[и:ε]--> q4   FINAL
```


In [ ]:
verb_alt_s_sh.show()
verb_alt_b_bl.show()


In [ ]:
print('с→ш exceptions (-сити → -шу):')
test(verb_alt_s_sh, 'голосити',  'голошу')    # to proclaim
test(verb_alt_s_sh, 'виносити',  'виношу')    # to carry out
test(verb_alt_s_sh, 'заносити',  'заношу')    # to carry in

print('б→бл exceptions (-бити → -блю):')
test(verb_alt_b_bl, 'загубити',  'загублю')   # to lose
test(verb_alt_b_bl, 'ослабити',  'ослаблю')   # to weaken
test(verb_alt_b_bl, 'зрубити',   'зрублю')    # to fell / chop down
test(verb_alt_b_bl, 'доробити',  'дороблю')   # to finish making

print('Regular rule gives WRONG forms for exception verbs:')
for w in ['голосити', 'загубити']:
    print(f'  verb_1sg_yty({w}) = {verb_1sg_yty.apply(w)}  <- incorrect')


## 11. Full Test Suite

In [ ]:
all_tests = [
    # Case 1 — hard stem
    (noun_gen_hard,  'держава',    'держави'),
    (noun_gen_hard,  'людина',     'людини'),
    (noun_gen_hard,  'хвилина',    'хвилини'),
    (noun_gen_hard,  'справа',     'справи'),
    (noun_gen_hard,  'садиба',     'садиби'),
    (noun_gen_hard,  'влада',      'влади'),
    # Case 1 — soft stem
    (noun_gen_soft,  'криниця',    'криниці'),
    (noun_gen_soft,  'лікарня',    'лікарні'),
    (noun_gen_soft,  'пустеля',    'пустелі'),
    (noun_gen_soft,  'знаряддя',   'знарядді'),
    (noun_gen_soft,  'праця',      'праці'),
    (noun_gen_soft,  'пекерня',    'пекерні'),
    # Case 2 — hard stem
    (adj_fem_hard,   'незалежний', 'незалежна'),
    (adj_fem_hard,   'відважний',  'відважна'),
    (adj_fem_hard,   'загадковий', 'загадкова'),
    (adj_fem_hard,   'оксамитовий','оксамитова'),
    (adj_fem_hard,   'бурштиновий','бурштинова'),
    (adj_fem_hard,   'зухвалий',   'зухвала'),
    # Case 2 — soft stem
    (adj_fem_soft,   'майбутній',   'майбутня'),
    (adj_fem_soft,   'сусідній',    'сусідня'),
    (adj_fem_soft,   'справжній',   'справжня'),
    (adj_fem_soft,   'безкрайній',  'безкрайня'),
    (adj_fem_soft,   'всесвітній',  'всесвітня'),
    (adj_fem_soft,   'довколишній', 'довколишня'),
    # Case 3 — -ати/-яти
    (verb_1sg_aty,   'відчувати',  'відчуваю'),
    (verb_1sg_aty,   'перемагати', 'перемагаю'),
    (verb_1sg_aty,   'зупиняти',   'зупиняю'),
    (verb_1sg_aty,   'кохати',     'кохаю'),
    (verb_1sg_aty,   'мріяти',     'мріяю'),
    (verb_1sg_aty,   'надихати',   'надихаю'),
    # Case 3 — -ити
    (verb_1sg_yty,   'цінити',     'ціню'),
    (verb_1sg_yty,   'хвалити',    'хвалю'),
    (verb_1sg_yty,   'молити',     'молю'),
    # Rule S
    (noun_dat_subst, 'перемога',   'перемозі'),
    (noun_dat_subst, 'наснага',    'насназі'),
    (noun_dat_subst, 'звага',      'звазі'),
    (noun_dat_subst, 'розвага',    'розвазі'),
    (noun_dat_subst, 'вимога',     'вимозі'),
    (noun_dat_subst, 'знемога',    'знемозі'),
    # Rule I
    (verb_reflexive, 'навчати',    'навчатися'),
    (verb_reflexive, 'намагати',   'намагатися'),
    (verb_reflexive, 'піклувати',  'піклуватися'),
    # Rule D
    (verb_stem,      'відчувати',  'відчува'),
    (verb_stem,      'співати',    'співа'),
    (verb_stem,      'обіймати',   'обійма'),
    (verb_stem,      'надихати',   'надиха'),
    (verb_stem,      'допомагати', 'допомага'),
    (verb_stem,      'захищати',   'захища'),
    # Complex — с→ш
    (verb_alt_s_sh,  'голосити',   'голошу'),
    (verb_alt_s_sh,  'виносити',   'виношу'),
    (verb_alt_s_sh,  'заносити',   'заношу'),
    # Complex — б→бл
    (verb_alt_b_bl,  'загубити',   'загублю'),
    (verb_alt_b_bl,  'ослабити',   'ослаблю'),
    (verb_alt_b_bl,  'зрубити',    'зрублю'),
    (verb_alt_b_bl,  'доробити',   'дороблю'),
]

passed = sum(1 for fst, w, e in all_tests if fst.apply(w) == e)
print(f'Results: {passed}/{len(all_tests)} tests passed')
print()
for fst, word, exp in all_tests:
    test(fst, word, exp)
